# laya-guard: fine-tune Laya on a free GPU

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Upload only `train/train.jsonl` and `train/val.jsonl` from the laya-guard checkout.
3. Download `laya-guard-ckpt.zip` at the end and unzip it next to the repo (`train/ckpt/`).

The two files hold masked commands from personal projects. They live only in this session: nothing here saves them to
Drive or a dataset. The checkpoint is trained on them too, so keep it private.

In [ ]:
# 1. A GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. The same laya version laya-guard uses locally, so the checkpoint loads there
!pip install -q laya==0.4.3
!laya-train --help | head -5

In [ ]:
# 3. Upload train.jsonl and val.jsonl
from google.colab import files
up = files.upload()
for name in ("train.jsonl", "val.jsonl"):
    assert name in up, f"missing {name}"
    print(name, sum(1 for line in open(name) if line.strip()), "rows")

In [ ]:
# 4. Check the data without loading the model
!laya-train --data train.jsonl --eval val.jsonl --base multilingual --out ckpt --dry-run

In [ ]:
# 5. Train (prints the validation score before and after). 2 epochs: the loss rose after the 2nd last time
!laya-train --data train.jsonl --eval val.jsonl --base multilingual --out ckpt --device cuda --seed 0 --epochs 2 2>&1 | tee train.log

In [ ]:
# 6. Zip the checkpoint and the log, and download them
import shutil, os
shutil.copy("train.log", "ckpt/train.log")
shutil.make_archive("laya-guard-ckpt", "zip", "ckpt")
print(round(os.path.getsize("laya-guard-ckpt.zip") / 2**20), "MB")
files.download("laya-guard-ckpt.zip")

Optional, instead of downloading: push to a **private** Hugging Face repo.

```python
from huggingface_hub import notebook_login, HfApi
notebook_login()                      # paste a write token in the box, never in a cell
api = HfApi()
repo = api.create_repo("<your-name>/laya-guard-ckpt", private=True, exist_ok=True).repo_id
api.upload_folder(folder_path="ckpt", repo_id=repo)
```